1. Doc2Vec 이용하여 ratings_train 학습
2. 학습된 모델 저장
3. 모델 로드, ratings_test 데이터 벡터화
    - 토큰화
    - infer_vector() 이용
4. LSTM 모델 이용하여 학습, 검증

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from konlpy.tag import Komoran
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

In [32]:
# 모델 학습, 저장

df = pd.read_csv('../data/ratings_train.txt', sep = '\t')
df = df[:100]

komoran = Komoran()

def tokenize(text):
    allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']
    result = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            result.append(word)
    return result

In [33]:
tokenized_sentence = [ tokenize(text) for text in df['document'] ]

In [34]:
tagged_data = [
    TaggedDocument(words = doc, tags = [str(i)]) for i, doc in enumerate(tokenized_sentence)
]

In [35]:
d2v = Doc2Vec(
    tagged_data, vector_size = 64, window = 5, min_count = 1, workers = 2, epochs = 20
)

In [36]:
d2v.save('my_model.model')

In [37]:
loaded_doc2vec = Doc2Vec.load('my_doc2vec.model')

In [49]:
df = pd.read_csv('../data/ratings_test.txt', sep = '\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        50000 non-null  int64
 1   document  49997 non-null  str  
 2   label     50000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.1 MB


In [50]:
df.dropna(inplace=True)
df.drop_duplicates('document', inplace = True)

In [51]:
df = df[:5000]

In [52]:
tokenized_sentence = [ tokenize(text) for text in df['document'] ]

In [53]:
X_vector = []

for sentence in tokenized_sentence:
    vec = loaded_doc2vec.infer_vector(sentence)
    X_vector.append(vec)

X = np.array(X_vector)
y = df['label'].values

In [54]:
class LSTMDataset(Dataset):
    def __init__( self, vectors, labels ):
        self.labels = labels
        self.data = vectors
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype = torch.float32), \
            torch.tensor(self.labels[idx], dtype = torch.long)

In [55]:
# Dataset 생성
dataset = LSTMDataset(X, y)

train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, test_size])

In [56]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 64, shuffle = True)

In [57]:
class LSTMCLF(nn.Module):

    def __init__(self, input_dim, hidden_size, num_classes, dropout = 0.5, head_type = 'last'):
        super().__init__()

        self.head_type = head_type
        # input_dim : 벡터화 데이터의 차원의 수
        self.lstm = nn.LSTM(input_dim, hidden_size, batch_first = True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)
    

    def forward(self, x):
        # 차원 확장
        x = x.unsqueeze(1)      # [ batch_size, 1, input_dim ]
        lstm_out, (hidden, cell) = self.lstm(x)

        if self.head_type == 'last':
            last_hidden = hidden.squeeze(0)
        elif self.head_type == 'mean':
            # 모든 층의 값들의 평균을 구한다.
            # lstm_out → [batch_size, seq_len, hidden_size]
            last_hidden = torch.mean(lstm_out, dim = 1)     # [batch_size, hidden_size]
        elif self.head_type == 'max':
            last_hidden, _ = torch.max(lstm_out, dim = 1)

        dropout_hidden = self.dropout(last_hidden)

        return self.fc(dropout_hidden)

In [58]:
# 모델 생성
model = LSTMCLF(input_dim = 64, hidden_size = 128, num_classes = 2, head_type = 'last', dropout = 0.3)

# 손실 함수
criterion = nn.CrossEntropyLoss()

# 옵티마이저
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [59]:
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct_train = 0
    total_train = 0

    for inputs, labels in tqdm(train_loader, desc = f'Epoch {epoch+1} / {epochs} Train'):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
    
        train_loss += loss.item()
        pred = torch.argmax(output, dim = 1)
        correct_train += (pred == labels).sum().item()
        total_train += labels.size(0)
    
    train_acc = (correct_train / total_train) * 100
    avg_train_loss = train_loss / len(train_loader)


    # 검증 구간
    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            output = model(inputs)
            loss = criterion(output, labels)
            val_loss += loss.item()
            pred = torch.argmax(output, dim = 1)
            correct_val += (pred == labels).sum().item()
            total_val += labels.size(0)
    
    val_acc = (correct_val / total_val) * 100
    avg_val_loss = val_loss / len(val_loader)

    if (epoch+1) % 10 == 0:
        print(f'RNN epoch - Train Loss: {round(avg_train_loss, 4)} / Train Acc : {train_acc}')
        print(f'RNN epoch - Val Loss: {round(avg_val_loss, 4)} / Val Acc : {val_acc}')

Epoch 10 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 156.70it/s]


RNN epoch - Train Loss: 0.5296 / Train Acc : 72.65
RNN epoch - Val Loss: 0.5239 / Val Acc : 73.5


Epoch 20 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 192.88it/s]


RNN epoch - Train Loss: 0.5188 / Train Acc : 73.6
RNN epoch - Val Loss: 0.5202 / Val Acc : 73.7


Epoch 30 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 193.54it/s]


RNN epoch - Train Loss: 0.5063 / Train Acc : 73.97500000000001
RNN epoch - Val Loss: 0.5227 / Val Acc : 73.2


Epoch 40 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 188.87it/s]


RNN epoch - Train Loss: 0.4898 / Train Acc : 75.02499999999999
RNN epoch - Val Loss: 0.5294 / Val Acc : 73.1


Epoch 50 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 192.81it/s]

RNN epoch - Train Loss: 0.4761 / Train Acc : 75.47500000000001
RNN epoch - Val Loss: 0.5298 / Val Acc : 73.3
